In [1]:
import sys
sys.path.append("/Users/avelezxerenity/Documents/GitHub/pysdk")
#sys.path.append("/Users/andre/Documents/xerenity/pysdk")
import numpy as np
import numpy_financial as npf
from scipy.optimize import newton
import os
import json
from src.xerenity.xty import Xerenity
from server.loan_calculator.loan_calculator import LoanCalculatorServer
from utilities.date_functions import datetime_to_ql,ql_to_datetime, calculate_irr
import pandas as pd
import QuantLib as ql
from loans_calculator.funciones_analisis_credito import merge_two_resulting_cashflows,create_cashflows_and_total_value,calculate_debt_duration
from  datetime import datetime
from utilities.date_functions import days_30_360_dt,days_30_360_ql,days_act_act_dt,days_act_act_ql,days_act_365_ql,days_act_365_dt
xty = Xerenity(
    username=os.getenv('XTY_USER'),
    password=os.getenv('XTY_PWD'),
)

all_loans_data = xty.get_all_loan_data(
    filter_date="2024-08-23"
)


ibr_cashflow = xty.get_ibr_data(
    loan_id="beb65020-e940-4995-9e1e-2f2208cdc638",
    filter_date="2024-08-23"
)
#calc = LoanCalculatorServer(ibr_cashflow, local_dev=True)
#loan_payments = calc.cash_flow_ibr()


# Define the value_date
value_date=datetime_to_ql(datetime.strptime(all_loans_data['filter_date'], '%Y-%m-%d'))
db_info=all_loans_data['db_info']

2024-09-11 08:19:03,497:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/auth/v1/token?grant_type=password "HTTP/1.1 200 OK"
2024-09-11 08:19:03,947:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/rpc/ibr_cash_flow_data_all "HTTP/1.1 200 OK"
2024-09-11 08:19:04,341:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/rpc/ibr_cash_flow_data "HTTP/1.1 200 OK"


In [2]:
days_converter_dir={'por_dias_360':'30/360','por_dias_365':'actual/365'}
db_info=all_loans_data['db_info']

In [3]:

# Initialize an empty dictionary to store results
results = {}

# Define a unique identifier for each loan (e.g., a code or index)
for i, loan in enumerate(all_loans_data['loans']):
    #try:
        # Create a copy of the current loan dictionary
        loan_temp = loan.copy()
        
        # Add or update 'db_info' in the copied dictionary
        loan_temp['db_info'] = db_info
        
        # Instantiate the LoanCalculatorServer with the updated dictionary
        calc = LoanCalculatorServer(loan_temp, local_dev=True)
        
        # Calculate loan payments
        loan_payments = calc.cash_flow_ibr()
        
        # Create cash flows and total value
        variables = create_cashflows_and_total_value(pd.DataFrame(loan_payments),
            value_date,
            datetime.strptime(loan['start_date'], '%Y-%m-%d'),
            days_converter_dir[loan['days_count']]
        )
        
                
        # Remove 'db_info' from loan_temp for the final result
        loan_temp.pop('db_info', None)  # Use pop to safely remove 'db_info' if it exists
        print(variables)
        # Store the variables along with the rest of the loan data
        results[f'loan_{i}'] = {
            'variables': variables,
            'loan_data': loan_temp
        }
        
    # except Exception as e:
    #     # Handle any unexpected exceptions
    #     print(f"An unexpected error occurred: {e}")
    #     print(loan)

# Print results to verify


/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '8333.333333333334' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments
/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '83333333.33333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats 

{'cashflows':          date       payment
0  2024-08-23 -99602.861111
1  2024-10-03   9524.750000
2  2024-11-03   9362.999859
3  2024-12-03   9275.101672
4  2025-01-03   9121.471535
5  2025-02-03   9015.828082
6  2025-03-03   8972.364705
7  2025-04-03   8810.907870
8  2025-05-03   8724.900329
9  2025-06-03   8640.290219
10 2025-07-03   8568.273531
11 2025-08-03   8486.811776
12 2025-09-03   8410.072555, 'irr': 0.13877107964643912, 'duration': 0.44051040314873724, 'tenor': 1.0294318959616702, 'interest': 6913.7721337504745, 'df':          date  beginning_balance       rate   rate_tot      payment  \
0  2024-10-03      100000.000000  10.047000  14.297000  9524.750000   
1  2024-11-03       91666.666667   9.229271  13.479271  9362.999859   
2  2024-12-03       83333.333333   9.311464  13.561464  9275.101672   
3  2025-01-03       75000.000000   8.360211  12.610211  9121.471535   
4  2025-02-03       66666.666667   8.034905  12.284905  9015.828082   
5  2025-03-03       58333.333333   8.89

/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 't'] = (df['date'] - df['date'].iloc[0]).dt.days / 365.25
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:144: SettingWithCopyWarnin

{'cashflows':          date       payment
0  2024-08-23 -9.479590e+08
1  2024-09-14  3.988349e+07
2  2024-10-14  3.976019e+07
3  2024-11-14  3.839277e+07
4  2024-12-14  3.858561e+07
5  2025-01-14  3.735728e+07
6  2025-02-14  3.614181e+07
7  2025-03-14  3.725721e+07
8  2025-04-14  3.536025e+07
9  2025-05-14  3.546185e+07
10 2025-06-14  3.479858e+07
11 2025-07-14  3.487076e+07
12 2025-08-14  3.445222e+07
13 2025-09-14  3.395608e+07
14 2025-10-14  3.363148e+07
15 2025-11-14  3.306903e+07
16 2025-12-14  3.306298e+07
17 2026-01-14  3.269864e+07
18 2026-02-14  3.197550e+07
19 2026-03-14  3.239809e+07
20 2026-04-14  3.174621e+07
21 2026-05-14  3.167214e+07
22 2026-06-14  3.121709e+07
23 2026-07-14  3.111580e+07
24 2026-08-14  3.078497e+07
25 2026-09-14  3.042340e+07
26 2026-10-14  3.042092e+07
27 2026-11-14  3.001051e+07
28 2026-12-14  2.983355e+07
29 2027-01-14  2.950814e+07
30 2027-02-14  2.908020e+07
31 2027-03-14  2.899934e+07
32 2027-04-14  2.861505e+07
33 2027-05-14  2.836514e+07
34 202

/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '166666666.66666666' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc

...

/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '69444444.44444445' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

{'cashflows':          date       payment
0  2024-08-23 -1.815996e+09
1  2024-09-09  9.619389e+07
2  2024-10-09  9.574359e+07
3  2024-11-09  9.281085e+07
4  2024-12-09  9.284230e+07
5  2025-01-09  9.033600e+07
6  2025-02-09  8.730225e+07
7  2025-03-09  8.929215e+07
8  2025-04-09  8.650293e+07
9  2025-05-09  8.614245e+07
10 2025-06-09  8.470054e+07
11 2025-07-09  8.427998e+07
12 2025-08-09  8.334875e+07
13 2025-09-09  8.199696e+07
14 2025-10-09  8.148628e+07
15 2025-11-09  8.019457e+07
16 2025-12-09  7.962381e+07
17 2026-01-09  7.869258e+07
18 2026-02-09  7.695027e+07
19 2026-03-09  7.683011e+07
20 2026-04-09  7.568860e+07
21 2026-05-09  7.496764e+07
22 2026-06-09  7.388621e+07
23 2026-07-09  7.310517e+07
24 2026-08-09  7.217394e+07
25 2026-09-09  7.118263e+07
26 2026-10-09  7.031147e+07, 'irr': 0.17696353706225998, 'duration': 0.9559056766874547, 'tenor': 2.127310061601643, 'interest': 393095840.0263283, 'df':          date  beginning_balance       rate   rate_tot       payment  \
0  2

/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '69380237.75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in

{'cashflows':          date       payment
0  2024-08-23 -5.064712e+08
1  2024-08-28  7.765417e+06
2  2024-09-28  7.749167e+06
3  2024-10-28  7.554526e+06
4  2024-11-28  7.296289e+06
..        ...           ...
91 2032-02-28  1.090202e+07
92 2032-03-28  1.078100e+07
93 2032-04-28  1.062285e+07
94 2032-05-28  1.048775e+07
95 2032-06-28  1.034367e+07

[96 rows x 2 columns], 'irr': 0.1709179929700697, 'duration': 4.1590875512391925, 'tenor': 7.846680355920602, 'interest': 472129073.5668564, 'df':          date  beginning_balance       rate   rate_tot       payment  \
0  2024-07-28       5.000000e+08  10.538000  19.038000  7.932500e+06   
1  2024-08-28       5.000000e+08  10.137000  18.637000  7.765417e+06   
2  2024-09-28       5.000000e+08  10.098000  18.598000  7.749167e+06   
3  2024-10-28       5.000000e+08   9.630863  18.130863  7.554526e+06   
4  2024-11-28       5.000000e+08   9.011094  17.511094  7.296289e+06   
..        ...                ...        ...        ...           ...  

/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '83333333.33333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

In [4]:
results['loan_0']

{'variables': {'cashflows':          date       payment
  0  2024-08-23 -99602.861111
  1  2024-10-03   9524.750000
  2  2024-11-03   9362.999859
  3  2024-12-03   9275.101672
  4  2025-01-03   9121.471535
  5  2025-02-03   9015.828082
  6  2025-03-03   8972.364705
  7  2025-04-03   8810.907870
  8  2025-05-03   8724.900329
  9  2025-06-03   8640.290219
  10 2025-07-03   8568.273531
  11 2025-08-03   8486.811776
  12 2025-09-03   8410.072555,
  'irr': 0.13877107964643912,
  'duration': 0.44051040314873724,
  'tenor': 1.0294318959616702,
  'interest': 6913.7721337504745,
  'df':          date  beginning_balance       rate   rate_tot      payment  \
  0  2024-10-03      100000.000000  10.047000  14.297000  9524.750000   
  1  2024-11-03       91666.666667   9.229271  13.479271  9362.999859   
  2  2024-12-03       83333.333333   9.311464  13.561464  9275.101672   
  3  2025-01-03       75000.000000   8.360211  12.610211  9121.471535   
  4  2025-02-03       66666.666667   8.034905  12.28

In [5]:
# Two different loans
loan0_df=results['loan_0']
loan1_df=results['loan_1']

In [10]:
# new loan dict
new_loan={'variables':{}}

# Methos to add cashflows. 
new_loan['variables']['cashflows']=merge_two_resulting_cashflows(loan0_df['cashflows'],loan1_df['cashflows'])
new_loan['variables']['irr']=calculate_irr(new_loan['variables']['cashflows']['date'],new_loan['variables']['cashflows']['payment'],'30/360')
new_loan['variables']['duration']=calculate_debt_duration(new_loan['variables']['cashflows'][new_loan['variables']['cashflows']['payment']>0],new_loan['variables']['irr'])
new_loan['variables']['interest']=loan0_df['variables']['interest']+loan1_df['variables']['interest']
new_loan['variables']['total_value']=loan0_df['variables']['total_value']+loan1_df['variables']['total_value']
new_loan['variables']['principal_out_value']=loan0_df['variables']['principal_out_value']+loan1_df['variables']['principal_out_value']

/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 't'] = (df['date'] - df['date'].iloc[0]).dt.days / 365.25
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:144: SettingWithCopyWarnin

KeyError: 'variables'

In [ ]:
calculate_debt_duration()

In [ ]:

new_loan['cashflows']

In [ ]:
new_loan['irr']

In [ ]:
mergerd_cf

In [ ]:
loan_payments = calc.cash_flow_ibr()

In [ ]:
variables=create_cashflows_and_total_value(pd.DataFrame(loan_payments), value_date, days_converter_dir[loan['days_count']])

In [ ]:
variables

In [ ]:
#
# for loan in all_loans_data['loans']:

calc = LoanCalculatorServer(loan, local_dev=True)
loan_payments = calc.cash_flow_ibr()
variables=create_cashflows_and_total_value(pd.DataFrame(loan_payments), value_date, days_converter_dir[loan['days_count']])

In [ ]:
db_info

In [ ]:
isinstance(db_info, list)

In [ ]:
\





In [ ]:

variables


In [ ]:
type(variables)

In [ ]:
create_cashflows_and_total_value(loan_payments, value_date, '30/360')

In [ ]:
calculate_debt_duration(loan_payments)

In [ ]:
flows.to_clipboard()

In [ ]:
calculate_irr(flows['date'],flows['payment'],'30/360')*100

In [ ]:
loan_payments.to_clipboard()